# 1. Import thư viện và các bảng dữ liệu

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
from matplotlib.ticker import FuncFormatter
from matplotlib.ticker import StrMethodFormatter

import os

input_dir='datasets/'
output_dir='output/'

orders=pd.read_csv(input_dir+'orders.csv')
order_items=pd.read_csv(input_dir+'order_items.csv')
payments=pd.read_csv(input_dir+'payments.csv')
customers=pd.read_csv(input_dir+'customers.csv')
returns=pd.read_csv(input_dir+'returns.csv')
review=pd.read_csv(input_dir+'reviews.csv')
products=pd.read_csv(input_dir+'products.csv')
geography=pd.read_csv(input_dir+'geography.csv')

# 2. Ghép các bảng với thông tin liên quan lại thành bảng master

In [ ]:
master = pd.merge(orders, order_items, on='order_id', how='left')
master = pd.merge(master, payments, on='order_id', how='left')
master = pd.merge(master, customers, on='customer_id', how='left')

master = master.rename(columns={'zip_x': 'zip', 'payment_method_x': 'payment_method'})
master = master.drop(columns=['zip_y', 'payment_method_y'])

master = pd.merge(master, geography, on='zip', how='left')

master = master.rename(columns={'city_y': 'city'})
master = master.drop(columns=['city_x'])

master = pd.merge(master, returns, on=['order_id', 'product_id'], how='left', suffixes=('', '_ret'))
master = pd.merge(master, review, on=['order_id', 'product_id', 'customer_id'], how='left')

In [ ]:
master['order_date'] = pd.to_datetime(master['order_date'])
snapshot_date=master['order_date'].max()+dt.timedelta(days=1)

metrics = master.groupby('customer_id').agg({
    'order_date': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'nunique',
    'payment_value': 'sum',
    'refund_amount': 'sum',
    'rating': 'mean'
}).rename(columns={'order_date': 'recency', 'order_id': 'frequency'})

metrics['net_monetary']=metrics['payment_value']-metrics['refund_amount'].fillna(0)

# 3. Phân tích và đánh giá

## 3.1. Biểu đồ Pareto
Biểu đồ giúp xác định nhóm khách hàng trọng tâm. Từ biểu đồ trên, ta thấy
- Khoảng 20% khách hàng đóng góp vào gần 62% doanh thu
- 80% doanh thu của doanh nghiệp được đóng góp bởi gần 40% khách hàng

In [ ]:
df_pareto = metrics.sort_values('net_monetary', ascending=False)
df_pareto['cum_sum'] = df_pareto['net_monetary'].cumsum()
df_pareto['cum_perc'] = 100 * df_pareto['cum_sum'] / df_pareto['net_monetary'].sum()
df_pareto['cust_perc'] = 100 * (pd.Series(range(1, len(df_pareto) + 1)) / len(df_pareto)).values

plt.figure(figsize=(10, 6))
sns.lineplot(x='cust_perc', y='cum_perc', data=df_pareto)
plt.axhline(80, color='r', linestyle='--')
plt.axvline(20, color='r', linestyle='--')
plt.title('Pareto Analysis: Revenue Concentration')
plt.xlabel('% of Customers')
plt.ylabel('% of Total Revenue')
plt.savefig(output_dir+"pareto_chart.png", dpi=300, bbox_inches='tight')
plt.show()

## 3.2. Phân tích top 10 khách hàng chi trả nhiều nhất

In [ ]:
ranked=metrics.sort_values('net_monetary', ascending=False)
cutoff_index=int(len(ranked)*0.20)
top_ids = metrics.sort_values('net_monetary', ascending=False).head(cutoff_index).index
metrics['is_hvc'] = metrics.index.isin(top_ids)

vip_list=ranked.head(cutoff_index).copy()
final_hvc_report=pd.merge(vip_list, customers, on='customer_id')
final_hvc_report=pd.merge(final_hvc_report, geography, on='zip')
final_hvc_report=final_hvc_report.drop(columns=['city_y'])
final_hvc_report=final_hvc_report.rename(columns={'city_x': 'city'})

top10 = final_hvc_report[['customer_id', 'net_monetary', 'city']].head(10)

fig, ax = plt.subplots(figsize=(10, 6))

plt.subplots_adjust(left=0.2) 

y_positions = range(len(top10))
bars = ax.barh(y_positions, top10['net_monetary'])

ax.set_yticks(y_positions)
ax.set_yticklabels([]) 

for i, (_, row) in enumerate(top10.iterrows()):
    ax.text(-0.02, i - 0.15, str(row['customer_id']), 
            transform=ax.get_yaxis_transform(), 
            ha='right', va='center', fontsize=11, fontweight='bold')
    
    ax.text(-0.02, i + 0.25, row['city'], 
            transform=ax.get_yaxis_transform(), 
            ha='right', va='center', fontsize=8, color='dimgray')

ax.xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))

ax.bar_label(bars, fmt='{:,.1f}', padding=5)

ax.grid(True, axis='x', linestyle='--', alpha=0.7)
ax.set_axisbelow(True)

ax.set_xlabel('Tổng số tiền')
ax.set_title('Top 10 Khách Hàng Chi Tiêu Nhiều Nhất (2022–2024)')

ax.text(
    1.0, -0.1, 'Đơn vị: VND',
    transform=ax.transAxes,
    ha='right', va='top',
    fontsize=10,
    fontstyle='italic',
    color='dimgray'
)

ax.invert_yaxis() 

ax.set_xlim(right=top10['net_monetary'].max() * 1.15) 
plt.savefig(output_dir+"top_10_monetary.png", dpi=300, bbox_inches='tight')
plt.show()

## 3.3. So sánh giữa phần trăm khách hàng và phần trăm doanh thu

In [ ]:
hvc_flat = metrics.reset_index()

gap_data = hvc_flat.groupby('is_hvc').agg({
    'customer_id': 'count',      # Now this column exists!
    'net_monetary': 'sum'
}).rename(columns={'customer_id': 'Customer_Count', 'net_monetary': 'Total_Revenue'})

gap_perc = gap_data.div(gap_data.sum(), axis=1) * 100
gap_perc.index = ['Non-HVC', 'HVC']

ax = gap_perc.plot(kind='bar', figsize=(10, 6), width=0.8, color=['#b2bec3', '#00b894'])

plt.title('20% Khách Hàng HVC Đóng Góp 61.5% Doanh Thu Từ 2012-2022')
plt.ylabel('Tỷ lệ (%)')
plt.xticks(rotation=0)
plt.legend(['Phần trăm khách hàng (%)', 'Phần trăm doanh thu (%)'])

for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width() / 2., p.get_height() + 1), 
                ha='center', weight='bold')

plt.tight_layout()
plt.savefig(output_dir+"hvc_pareto.png", dpi=300, bbox_inches='tight')
plt.show()

## 3.4. Phân tích tỉ lệ các khách hàng tiềm năng (HVC) qua các kênh tiếp cận, giới tính và độ tuổi

In [ ]:
full_profile = pd.merge(customers, metrics[['is_hvc']], on='customer_id', how='left')
full_profile['is_hvc'] = (
    full_profile['is_hvc']
    .astype('boolean')
    .fillna(False)
)


channel_analysis = full_profile.pivot_table(
    index='acquisition_channel', 
    columns='is_hvc', 
    values='customer_id', 
    aggfunc='count', 
    fill_value=0
)

channel_analysis.columns = ['Standard_Cust', 'HVC_Cust']
channel_analysis['HVC_Rate_%'] = (channel_analysis['HVC_Cust'] / (channel_analysis['HVC_Cust'] + channel_analysis['Standard_Cust'])) * 100

channel_mapping = {
    'paid_search': 'Tìm kiếm trả phí',
    'social_media': 'Mạng Xã hội',
    'organic_search': 'Tìm kiếm tự nhiên',
    'direct': 'Truy cập trực tiếp',
    'email_campaign': 'Email',
    'referral': 'Qua giới thiệu'
}

channel_analysis = channel_analysis.rename(index=channel_mapping)

channel_sorted = channel_analysis.sort_values('HVC_Rate_%', ascending=True)

plt.figure()
plt.barh(channel_sorted.index, channel_sorted['HVC_Rate_%'])

plt.xlabel('Tỷ lệ HVC (%)')
plt.ylabel('Kênh bán hàng')
plt.title('Tỷ Lệ HVC Qua Các Kênh Bán Hàng (2012-2022)')
max_val = channel_sorted['HVC_Rate_%'].max()
plt.xlim(0, max_val * 1.2)
# Add labels
for i, v in enumerate(channel_sorted['HVC_Rate_%']):
    plt.text(v, i, f" {v:.1f}%", va='center')

plt.savefig(output_dir+"hvc_channel.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
age_analysis = full_profile.pivot_table(
    index='age_group',
    columns='is_hvc',
    values='customer_id',
    aggfunc='count',
    fill_value=0
)
age_analysis.columns = ['Standard', 'HVC']
age_analysis['HVC_Concentration_%'] = (age_analysis['HVC'] / age_analysis['HVC'].sum()) * 100
gender_analysis = full_profile.pivot_table(
    index='gender', 
    columns='is_hvc', 
    values='customer_id', 
    aggfunc='count', 
    fill_value=0
)
gender_analysis.columns = ['Standard', 'HVC']
gender_analysis['HVC_Rate'] = gender_analysis['HVC'] / (gender_analysis['HVC'] + gender_analysis['Standard'])
gender_mapping = {
    'Male' : 'Nam',
    'Female': 'Nữ',
    'Non-binary': 'Phi nhị giới'
}

gender_analysis = gender_analysis.rename(index=gender_mapping)

gender_plot = gender_analysis.copy()

plt.figure()
plt.bar(gender_plot.index.astype(str), gender_plot['HVC_Rate'])

plt.xlabel('Giới tính')
plt.ylabel('Tỷ lệ HVC (%)')
plt.title('Tỷ Lệ HVC theo Giới Tính (2012-2022)')

for i, v in enumerate(gender_plot['HVC_Rate']):
    plt.text(i, v, f"{v:.2%}", ha='center', va='bottom')

plt.tight_layout()
plt.show()

age_order = ['18-24', '25-34', '35-44', '45-54', '55+']
age_plot = age_analysis.reindex(age_order)

top_2_threshold = age_plot['HVC_Concentration_%'].nlargest(2).min()

colors = ['#1f77b4' if v >= top_2_threshold else 'lightgray' for v in age_plot['HVC_Concentration_%']]

plt.figure(figsize=(10, 6))

bars = plt.barh(age_plot.index.astype(str), age_plot['HVC_Concentration_%'], color=colors)

plt.xlabel('Tỉ lệ HVC (%)')
plt.ylabel('Nhóm tuổi')
plt.title('Tỷ Lệ HVC theo Nhóm Tuổi (2012-2022)')

max_val = age_plot['HVC_Concentration_%'].max()
plt.xlim(0, max_val * 1.2)
plt.gca().invert_yaxis()

for i, v in enumerate(age_plot['HVC_Concentration_%']):
    text_color = '#1f77b4' if v >= top_2_threshold else 'dimgray'
    plt.text(v, i, f" {v:.1f}%", va='center', color=text_color, 
             fontweight='bold' if v >= top_2_threshold else 'normal')

plt.tight_layout()
plt.savefig(output_dir+"hvc_gender.png", dpi=300, bbox_inches='tight')
plt.savefig(output_dir+"hvc_cohort.png", dpi=300, bbox_inches='tight')
plt.show()

## 3.5. Phân tích mật độ HVC và thị trường theo thành phố

In [ ]:
master['net_revenue'] = master['payment_value'] - master['refund_amount'].fillna(0)

city_revenue = master.groupby('city')['net_revenue'].sum().reset_index()

city_revenue = city_revenue.sort_values(by='net_revenue', ascending=False)


top_cities = city_revenue.head(10)

# Create figure and axis objects for better control
fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(top_cities['city'], top_cities['net_revenue'], color='#3498db')

# Labels and title
ax.set_xlabel('Thành phố')
ax.set_ylabel('Doanh thu')
ax.set_title('10 thành phố có doanh thu cao nhất')

# Rotate city names and align them to the right so they look cleaner
plt.xticks(rotation=45, ha='right')

# 1. FIX SCIENTIFIC NOTATION: Format y-axis with commas and no decimals
ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))

ax.annotate('Đơn vị: VND', xy=(1.0, -0.35), xycoords='axes fraction', 
            ha='right', va='center', fontsize=10, fontstyle='italic', color='dimgray')

# Adjust layout to make room for the rotated text and the bottom-right unit
plt.tight_layout()
plt.show()

In [ ]:
geo_profile=pd.merge(full_profile, geography, on='zip')
city_hvc_trend=geo_profile.groupby('city_x').agg({
    'is_hvc': 'sum',
    'customer_id': 'count'
}).rename(columns={'is_hvc': 'HVC_Count', 'customer_id': 'Total_Cust'})

city_hvc_trend['HVC_Density_%']=(city_hvc_trend['HVC_Count'])/(city_hvc_trend['Total_Cust'])*100

top_cities = city_hvc_trend.sort_values('HVC_Density_%', ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(top_cities.index.astype(str), top_cities['HVC_Density_%'], color='#00b894')

plt.gca().invert_yaxis() 

plt.xlabel('Mật độ HVC (%)')
plt.ylabel('Thành phố')
plt.title('Top Các Thành Phố dựa trên mật độ HVC')

max_val = city_hvc_trend['HVC_Density_%'].max()
plt.xlim(0, max_val * 1.2)

for i, v in enumerate(top_cities['HVC_Density_%']):
    plt.text(v + 0.5, i, f"{v:.1f}%", va='center', fontweight='bold')

plt.tight_layout()
plt.savefig(output_dir+"hvc_city_density.png", dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(12, 8))
plt.scatter(city_hvc_trend['HVC_Count'], city_hvc_trend['HVC_Density_%'], alpha=0.6, color='#00b894', s=60)
plt.xlabel('Số khách hàng HVC')
plt.ylabel('Mật độ HVC (%)')
plt.title('Phân tích thị trường HVC theo thành phố (2012-2022)')

median_count = city_hvc_trend['HVC_Count'].median()
median_density = city_hvc_trend['HVC_Density_%'].median()

min_x, max_x = plt.xlim()
min_y, max_y = plt.ylim()

plt.axvline(x=median_count, color='red', linestyle='--', alpha=0.5, label=f'Trung vị số khách hàng: {median_count:.0f}')
plt.axhline(y=median_density, color='red', linestyle='--', alpha=0.5, label=f'Trung vị mật độ: {median_density:.1f}%')

offset_x = (max_x - min_x) * 0.03
offset_y = (max_y - min_y) * 0.03

plt.text(min_x + offset_x, max_y - offset_y,
         'Tiềm năng nhỏ', color='darkgrey', fontsize=11, fontweight='bold', ha='left', va='top')
plt.text(max_x - offset_x, max_y - offset_y,
         'Thị trường vàng', color='darkgrey', fontsize=11, fontweight='bold', ha='right', va='top')
plt.text(max_x - offset_x, min_y + offset_y,
         'Dư địa phát triển', color='darkgrey', fontsize=11, fontweight='bold', ha='right', va='bottom')
plt.text(min_x + offset_x, min_y + offset_y,
         'Uư tiên thấp', color='darkgrey', fontsize=11, fontweight='bold', ha='left', va='bottom')

high_density = city_hvc_trend['HVC_Density_%'] > median_density
high_count   = city_hvc_trend['HVC_Count']     > median_count

quadrants = {
    'Golden':       (city_hvc_trend[ high_density &  high_count], 'darkgreen', 'HVC_Density_%'),
    'Potential':    (city_hvc_trend[ high_density & ~high_count], 'darkgreen',      'HVC_Density_%'),
    'Growth':       (city_hvc_trend[~high_density &  high_count], 'darkgreen',    'HVC_Count'),
    'Low Priority': (city_hvc_trend[~high_density & ~high_count], 'darkgreen',       'HVC_Count'),
}

for label, (df, color, sort_col) in quadrants.items():
    if not df.empty:
        top_cities = df.nlargest(2, sort_col)
        for idx, row in top_cities.iterrows():
            plt.annotate(
                idx,
                (row['HVC_Count'], row['HVC_Density_%']),
                textcoords="offset points", xytext=(6, 6),
                ha='left', fontsize=9, fontweight='bold', color=color
            )

plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=2)
plt.grid(True, which='both', linestyle=':', alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir + "hvc_city_quadrant.png", dpi=300, bbox_inches='tight')
plt.show()

## 3.6. Phân tích chi tiêu của các nhóm khách hàng trên từng quadrant về Danh mục (Category) và Phân khúc sản phẩm (Segment)

In [ ]:
median_count = city_hvc_trend['HVC_Count'].median()
median_density = city_hvc_trend['HVC_Density_%'].median()

def assign_quadrant(row):
    if row['HVC_Count'] > median_count and row['HVC_Density_%'] > median_density:
        return 'Thị trường Vàng (Golden)'
    elif row['HVC_Count'] <= median_count and row['HVC_Density_%'] > median_density:
        return 'Tiềm năng nhỏ (Potential)'
    elif row['HVC_Count'] > median_count and row['HVC_Density_%'] <= median_density:
        return 'Dư địa phát triển (Growth)'
    else:
        return 'Ưu tiên thấp (Low Priority)'

city_hvc_trend['Quadrant'] = city_hvc_trend.apply(assign_quadrant, axis=1)
quadrant_mapping = city_hvc_trend[['Quadrant']].reset_index()

quadrant_mapping = quadrant_mapping.rename(columns={'city_x': 'city', 'index': 'city'}) 

fashion_quadrant = fashion_fit.merge(quadrant_mapping, on='city', how='inner')

hvc_quadrant_data = fashion_quadrant[fashion_quadrant['is_hvc'] == True]

quadrant_segment_spend = hvc_quadrant_data.groupby(['Quadrant', 'segment'])['payment_value'].sum().reset_index()

pivot_df = quadrant_segment_spend.pivot(index='Quadrant', columns='segment', values='payment_value').fillna(0)

pivot_percentage = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 7))

pivot_percentage.plot(kind='bar', stacked=True, ax=ax, colormap='Set2', edgecolor='white', linewidth=1)

ax.set_xlabel('Góc phần tư (Quadrant)', fontsize=11)
ax.set_ylabel('Tỷ trọng chi tiêu (%)', fontsize=11)
ax.set_title('Tỷ trọng chi tiêu của HVC theo phân khúc ở từng Nhóm thị trường (2012-2022)', fontsize=14, pad=20)

plt.xticks(rotation=0)

ax.set_ylim(0, 100)

plt.legend(title='Phân khúc (Segment)', bbox_to_anchor=(1.05, 1), loc='upper left')

for c in ax.containers:
    labels = [f"{v.get_height():.1f}%" if v.get_height() > 3 else "" for v in c]
    ax.bar_label(c, labels=labels, label_type='center', fontsize=9, color='black')

plt.tight_layout()
plt.savefig(output_dir + "hvc_segment.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
median_count = city_hvc_trend['HVC_Count'].median()
median_density = city_hvc_trend['HVC_Density_%'].median()

def assign_quadrant(row):
    if row['HVC_Count'] > median_count and row['HVC_Density_%'] > median_density:
        return 'Thị trường Vàng (Golden)'
    elif row['HVC_Count'] <= median_count and row['HVC_Density_%'] > median_density:
        return 'Tiềm năng nhỏ (Potential)'
    elif row['HVC_Count'] > median_count and row['HVC_Density_%'] <= median_density:
        return 'Dư địa tăng trưởng (Growth)'
    else:
        return 'Ưu tiên thấp (Low Priority)'

city_hvc_trend['Quadrant'] = city_hvc_trend.apply(assign_quadrant, axis=1)
quadrant_mapping = city_hvc_trend[['Quadrant']].reset_index()
quadrant_mapping = quadrant_mapping.rename(columns={'city_x': 'city', 'index': 'city'}) 

fashion_quadrant = fashion_fit.merge(quadrant_mapping, on='city', how='inner')
hvc_quadrant_data = fashion_quadrant[fashion_quadrant['is_hvc'] == True]

quadrant_category_spend = hvc_quadrant_data.groupby(['Quadrant', 'category'])['payment_value'].sum().reset_index()
pivot_df = quadrant_category_spend.pivot(index='Quadrant', columns='category', values='payment_value').fillna(0)
pivot_percentage = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 7))
pivot_percentage.plot(kind='bar', stacked=True, ax=ax, colormap='tab20', edgecolor='white', linewidth=1)

ax.set_xlabel('Góc phần tư (Quadrant)', fontsize=11)
ax.set_ylabel('Tỷ trọng chi tiêu (%)', fontsize=11)
ax.set_title('Tỷ trọng chi tiêu HVC theo Danh mục ở từng Nhóm thị trường', fontsize=14, pad=20)

plt.xticks(rotation=0)
ax.set_ylim(0, 100)

plt.legend(title='Danh mục (Category)', bbox_to_anchor=(1.05, 1), loc='upper left')

for c in ax.containers:
    labels = [f"{v.get_height():.1f}%" if v.get_height() > 3 else "" for v in c]
    ax.bar_label(c, labels=labels, label_type='center', fontsize=9, color='black')

plt.tight_layout()
plt.show()